# Week 5 Experiments

This notebook runs the prompt-driven baseline required for the Week 5 milestone.

The project uses one Qwen-VL model for both roles:

- text-only prompt: works like the LLM stage
- text + sampled frames prompt: works like the VLM stage


In [ ]:
# Run this once in a fresh environment.
# !pip install -r ../requirements.txt


In [ ]:
from pathlib import Path
import json
import sys

# Let the notebook import from the project root.
PROJECT_ROOT = Path('..').resolve()
sys.path.append(str(PROJECT_ROOT))

from src.run_pipeline import run
from evaluation.metrics import evaluate_predictions


## 1. Choose a Video

Place a supervisor sample video in `data/videos/`, then update the path below.

In [ ]:
video_path = PROJECT_ROOT / 'data' / 'videos' / 'sample.mp4'
settings_path = PROJECT_ROOT / 'configs' / 'settings.yaml'

video_path


## 2. Run the Week 5 Baseline

This may take time because Qwen-VL loads a large model.

In [ ]:
result = run(str(video_path), str(settings_path))
result['video_id'], len(result['segments'])


In [ ]:
# Show the first segment in a readable way.
if result['segments']:
    print(json.dumps(result['segments'][0], indent=2)[:3000])


## 3. Evaluate Against Ground Truth

Create a ground-truth JSON file with this simple format:

```json
[
  {"start_time": 0.0, "end_time": 2.0, "action_label": "open_hand"}
]
```

In [ ]:
ground_truth_path = PROJECT_ROOT / 'data' / 'annotations' / f"{result['video_id']}_ground_truth.json"

if ground_truth_path.exists():
    ground_truth = json.loads(ground_truth_path.read_text(encoding='utf-8'))
    metrics = evaluate_predictions(result['segments'], ground_truth, iou_threshold=0.5)
else:
    metrics = {
        'status': 'missing_ground_truth',
        'expected_file': str(ground_truth_path),
        'prediction_segments': len(result['segments']),
    }

metrics


In [ ]:
# Save the Week 5 baseline result.
baseline_path = PROJECT_ROOT / 'evaluation' / 'baseline_results.json'
baseline_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
baseline_path
